In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [5]:
def prepare_features(df, max_score=100):
    feature_map = {
        'Previous_Scores': ['Previous_Scores', 'previous_score', 'prev_score', 'last_score', 'prior_score', 
                           'previous_grade', 'last_grade', 'past_score', 'PreviousScores', 'previous marks'],
        'Attendance': ['Attendance', 'attendance_percentage', 'attendance_rate', 'attendance_pct', 
                      'attend_rate', 'present_rate', 'presence', 'AttendanceRate', 'attendance %'],
        'Hours_Studied': ['Hours_Studied', 'daily_study_hours', 'study_hours', 'hours_studied', 
                         'study_time', 'study_hours_per_day', 'weekly_study_hours', 'HoursStudied', 'hours per day'],
        'Tutoring_Sessions': ['Tutoring_Sessions', 'tutoring_sessions', 'tutor_sessions', 'extra_classes', 
                             'tutoring', 'tutorial_sessions', 'TutoringSessions', 'extra help']
    }
    
    result = pd.DataFrame()
    for target, sources in feature_map.items():
        for source in sources:
            if source in df.columns:
                val = df[source]
                if target == 'Previous_Scores' and val.max() <= 20:
                    val = val * (max_score / 20)
                elif target == 'Previous_Scores' and val.max() <= 5:
                    val = val * (max_score / 5)
                result[target] = val
                break
        else:
            result[target] = 0
    return result

In [6]:
df1 = pd.read_csv("Data/StudentPerformanceFactors.csv")
df2 = pd.read_csv("Data/Student_performance_interactions.csv")
print(f"Dataset 1: {df1.shape}")
print(f"Dataset 2: {df2.shape}")

Dataset 1: (6607, 20)
Dataset 2: (1000, 18)


In [7]:
X1 = prepare_features(df1)
y1 = df1['Exam_Score']
X2_full = prepare_features(df2)
y2_full = df2['final_score']

X_combined = pd.concat([X1, X2_full], ignore_index=True)
y_combined = pd.concat([y1, y2_full], ignore_index=True)

X_train, X_test, y_train, y_test = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42)
print(f"Combined training data: {X_train.shape}")

Combined training data: (6085, 4)


In [8]:
class StudentPerformancePredictor:
    def __init__(self, model):
        self.model = model
    
    def prepare_features(self, df, max_score=100):
        feature_map = {
            'Previous_Scores': ['Previous_Scores', 'previous_score', 'prev_score', 'last_score', 'prior_score', 
                               'previous_grade', 'last_grade', 'past_score', 'PreviousScores', 'previous marks', 'grade_2'],
            'Attendance': ['Attendance', 'attendance_percentage', 'attendance_rate', 'attendance_pct', 
                          'attend_rate', 'present_rate', 'presence', 'AttendanceRate', 'attendance %', 'absences'],
            'Hours_Studied': ['Hours_Studied', 'daily_study_hours', 'study_hours', 'hours_studied', 
                             'study_time', 'study_hours_per_day', 'weekly_study_hours', 'HoursStudied', 'hours per day'],
            'Tutoring_Sessions': ['Tutoring_Sessions', 'tutoring_sessions', 'tutor_sessions', 'extra_classes', 
                                 'tutoring', 'tutorial_sessions', 'TutoringSessions', 'extra help', 'extra_paid_classes']
        }
        
        result = pd.DataFrame()
        for target, sources in feature_map.items():
            for source in sources:
                if source in df.columns:
                    val = df[source]
                    if target == 'Previous_Scores':
                        if source == 'grade_2':
                            val = val * 5
                        elif val.max() <= 20:
                            val = val * (max_score / 20)
                        elif val.max() <= 5:
                            val = val * (max_score / 5)
                    elif target == 'Attendance' and source == 'absences':
                        val = 100 - (val / val.max() * 100)
                    elif target == 'Hours_Studied' and source == 'study_time':
                        if val.dtype == 'object' or val.dtype == 'str' or pd.api.types.is_string_dtype(val):
                            val = val.map({'<2 hours': 1, '2 to 5 hours': 3.5, '5 to 10 hours': 7.5, '>10 hours': 12})
                    elif target == 'Tutoring_Sessions' and source == 'extra_paid_classes':
                        if val.dtype == 'object' or val.dtype == 'str' or pd.api.types.is_string_dtype(val):
                            val = val.map({'yes': 1, 'no': 0})
                    result[target] = val
                    break
            else:
                result[target] = 0
        return result
    
    def predict(self, df, output_max_score=100):
        X = self.prepare_features(df)
        pred = self.model.predict(X)
        if output_max_score != 100:
            pred = pred * (output_max_score / 100)
        return pred

In [9]:
df_math_prep = pd.read_csv("Data/student_math_clean.csv")
df_math_prep['Previous_Scores'] = df_math_prep['grade_2'] * 5
df_math_prep['Attendance'] = 100 - (df_math_prep['absences'] / df_math_prep['absences'].max() * 100)
df_math_prep['Hours_Studied'] = df_math_prep['study_time'].map({'<2 hours': 1, '2 to 5 hours': 3.5, '5 to 10 hours': 7.5, '>10 hours': 12})
df_math_prep['Tutoring_Sessions'] = df_math_prep['extra_paid_classes'].map({'yes': 1, 'no': 0})
X_math_train = df_math_prep[['Previous_Scores', 'Attendance', 'Hours_Studied', 'Tutoring_Sessions']]
y_math_train = df_math_prep['final_grade'] * 5

X_all = pd.concat([X_combined, X_math_train], ignore_index=True)
y_all = pd.concat([y_combined, y_math_train], ignore_index=True)

X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

model_new = RandomForestRegressor(n_estimators=800, max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=42, n_jobs=-1)
model_new.fit(X_train_new, y_train_new)

y_pred_new = model_new.predict(X_test_new)
print("=== Retrained Model Test Results ===")
print(f"MAE: {mean_absolute_error(y_test_new, y_pred_new):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_new, y_pred_new)):.4f}")
print(f"R²: {r2_score(y_test_new, y_pred_new):.4f}")

=== Retrained Model Test Results ===
MAE: 2.2157
RMSE: 3.8399
R²: 0.7883


In [ ]:
# Save ONLY the sklearn model (not the wrapper class)
joblib.dump(model_new, 'models/student_performance_rf.pkl')
print('Model saved to models/student_performance_rf.pkl!')

# Verify it loads correctly
loaded_model = joblib.load('models/student_performance_rf.pkl')
print(f"Model type: {type(loaded_model)}")
print(f"Model has predict method: {hasattr(loaded_model, 'predict')}")

New predictor saved to predictor_model.pkl!


In [11]:
predictor = joblib.load('predictor_model.pkl')

pred1 = predictor.predict(df1)
print(f"Dataset 1 - R²: {r2_score(y1, pred1):.4f}")

pred2 = predictor.predict(df2)
print(f"Dataset 2 - R²: {r2_score(y2_full, pred2):.4f}")

df_math = pd.read_csv('Data/student_math_clean.csv')
y_math = df_math['final_grade'] * 5
pred_math = predictor.predict(df_math)
print(f"Math Dataset - R²: {r2_score(y_math, pred_math):.4f}")


Dataset 1 - R²: 0.8542
Dataset 2 - R²: 0.9206
Math Dataset - R²: 0.9261
